# Лабораторная работа №3 «Оптимизация гиперпараметров»

In [24]:
import pandas as pd

Импорт датасетов из предыдущей лабораторной работы

In [25]:
small_df = pd.read_csv('../Lab2/ID1_S50_F5', sep = ",", encoding="utf-8")
small_df = small_df.drop(small_df.columns[0], axis=1)
large_df = pd.read_csv('../Lab2/ID12_S1500_F15', sep = ",", encoding="utf-8")
large_df = large_df.drop(large_df.columns[0], axis=1)

In [26]:
small_df.head(5)

,Obj1_bin_1,Obj1_nominal_2,Obj1_ordinal_3,Obj1_quantitative_4,Obj1_bin_5,Obj2_bin_1,Obj2_nominal_2,Obj2_ordinal_3,Obj2_quantitative_4,Obj2_bin_5,collision
0,1,1000,1,980.096113,0,0,500,3,977.171355,0,0
1,1,1000,1,62.853585,1,1,777,4,484.041589,1,1
2,1,500,2,106.014272,0,1,777,1,530.112921,1,0
3,1,1000,1,136.243390,1,1,777,1,105.784598,1,1
4,1,500,3,287.653502,1,1,1000,3,624.642187,0,0


In [27]:
large_df.head(5)

,Obj1_bin_1,Obj1_nominal_2,Obj1_ordinal_3,Obj1_quantitative_4,Obj1_bin_5,Obj1_nominal_6,Obj1_ordinal_7,Obj1_quantitative_8,Obj1_bin_9,Obj1_nominal_10,...,Obj2_ordinal_7,Obj2_quantitative_8,Obj2_bin_9,Obj2_nominal_10,Obj2_ordinal_11,Obj2_quantitative_12,Obj2_bin_13,Obj2_nominal_14,Obj2_ordinal_15,collision
0,0,500,2,179.369390,1,1000,1,220.479678,0,777,...,4,808.199044,0,500,3,221.703125,0,500,4,1
1,1,1000,3,186.354031,1,333,1,968.937961,1,500,...,4,630.463153,1,333,2,582.807334,0,333,4,1
2,1,1000,3,377.092703,0,777,3,567.345268,0,777,...,4,226.712704,1,333,2,304.761060,1,1000,4,0
3,1,1000,2,73.033068,0,777,1,855.936203,1,500,...,1,847.582755,0,333,1,340.000172,0,500,4,0
4,0,500,3,523.380040,0,333,2,826.869352,0,333,...,1,911.994991,0,333,4,182.358591,0,333,4,0


In [28]:
X_large = large_df.drop('collision', axis=1)
y_large = large_df['collision']

X_small = small_df.drop('collision', axis=1)
y_small = small_df['collision']

Лучшие модели из прошлой лабораторной работы

In [29]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

In [30]:
best_models = [
    KNeighborsClassifier(),
    RandomForestClassifier()
]

| Dataset | Model | F1 Score | Fit duration |
|---------|-------|----------|---------------|
| Small | KNeighborsClassifier | 0.600 | 0.610465 |
| Small | RandomForestClassifier | 0.680 | 4.116801 |
| Big | KNeighborsClassifier | 0.693 | 2.531045 |
| Big | RandomForestClassifier | 0.656 | 10.344154 |

# Задание №1 «Классический генетический алгоритм»

In [31]:
import pygad
import time
from sklearn.model_selection import cross_val_score
import joblib

In [32]:
STR_MAPS = {
    'weights': ['uniform', 'distance']
}

In [33]:
def fitness_func_factory(model_class, param_names, X, y):
    def fitness_func(ga_instance, solution, solution_idx):
        params = {}
        for name, val in zip(param_names, solution):
            if name in STR_MAPS:
                idx = int(val)
                params[name] = STR_MAPS[name][idx]
            else:
                params[name] = int(val)

        model = model_class(**params)
        score = cross_val_score(model, X, y, cv=3, scoring='f1_weighted', n_jobs=-1).mean()
        return score
    return fitness_func

In [34]:
def run_ga_optimization(model_class, param_space, X, y):
    param_names = list(param_space.keys())
    
    gene_space = []
    for name, space in param_space.items():
        if name in STR_MAPS:
            gene_space.append(range(len(STR_MAPS[name])))
        else:
            gene_space.append(space)
    
    fitness_function = fitness_func_factory(model_class, param_names, X, y)
    
    ga_instance = pygad.GA(
        num_generations=15,
        num_parents_mating=5,
        fitness_func=fitness_function,
        sol_per_pop=10,
        num_genes=len(param_names),
        gene_space=gene_space,
        parent_selection_type="sss",
        mutation_num_genes=1,
        stop_criteria=["reach_0.99"]
    )
    
    start_time = time.time()
    ga_instance.run()
    duration = time.time() - start_time
    
    solution, fitness, _ = ga_instance.best_solution()
    
    best_params = {}
    for name, val in zip(param_names, solution):
        if name in STR_MAPS:
            best_params[name] = STR_MAPS[name][int(val)]
        else:
            best_params[name] = int(val)
            
    return best_params, fitness, duration

In [35]:
knn_space = {
    'n_neighbors': range(1, 31),
    'weights': ['uniform', 'distance']
}

rf_space = {
    'n_estimators': range(10, 101, 10),
    'max_depth': range(2, 20)
}

In [36]:
datasets = {"Small": (X_small, y_small), "Large": (X_large, y_large)}
models = [("KNN", KNeighborsClassifier, knn_space), ("RF", RandomForestClassifier, rf_space)]

In [37]:
results_list = []

for ds_name, (X, y) in datasets.items():
    for m_name, m_class, m_space in models:
        params, score, duration = run_ga_optimization(m_class, m_space, X, y)
        
        final_model = m_class(**params).fit(X, y)
        model_filename = f"best_{m_name}_{ds_name}_gene"
        joblib.dump(final_model, model_filename)
        
        results_list.append({
            "Dataset": ds_name,
            "Model": m_name,
            "F1 Score": round(score, 4),
            "Duration": round(duration, 4),
            "Best Params": str(params)
        })

df_results = pd.DataFrame(results_list)
df_results

,Dataset,Model,F1 Score,Duration,Best Params
0,Small,KNN,0.7270,5.8326,"{'n_neighbors': 6, 'weights': 'uniform'}"
1,Small,RF,0.7151,4.7510,"{'n_estimators': 40, 'max_depth': 4}"
2,Large,KNN,0.5737,1.2660,"{'n_neighbors': 28, 'weights': 'uniform'}"
3,Large,RF,0.6202,10.5072,"{'n_estimators': 100, 'max_depth': 12}"


| Dataset | Model | F1 Score | Fit duration |
|---------|-------|----------|---------------|
| Small | KNeighborsClassifier | 0.600 | 0.610465 |
| Small | RandomForestClassifier | 0.680 | 4.116801 |
| Big | KNeighborsClassifier | 0.693 | 2.531045 |
| Big | RandomForestClassifier | 0.656 | 10.344154 |

# Задание №2 «Алгоритм роя частиц»

In [38]:
import numpy as np
import pyswarms as ps

In [39]:
def f1_eval(params, model_type, X, y):
    results = []
    for p in params:
        if model_type == 'knn':
            model = KNeighborsClassifier(n_neighbors=int(p[0]), weights=int(p[1]))
        else:
            model = RandomForestClassifier(n_estimators=int(p[0]), max_depth=int(p[1]), random_state=42)
        
        score = cross_val_score(model, X, y, cv=3, scoring='f1_weighted').mean()
        results.append(-score)
    return np.array(results)

In [40]:
def run_pso_tuning(model_type, X, y):
    if model_type == 'knn':
        lower_bound = np.array([1, 1])
        upper_bound = np.array([30, 2])
    else:
        lower_bound = np.array([10, 1])
        upper_bound = np.array([200, 20])
    
    bounds = (lower_bound, upper_bound)
    options = {'c1': 0.5, 'c2': 0.3, 'w': 0.9}
    
    optimizer = ps.single.GlobalBestPSO(n_particles=10, dimensions=2, options=options, bounds=bounds)
    cost, pos = optimizer.optimize(f1_eval, iters=10, model_type=model_type, X=X, y=y)
    
    return pos, -cost

In [42]:
datasets = [("Small", X_small, y_small), ("Large", X_large, y_large)]
models_to_tune = [('KNN', 'KNeighbors'), ('RF', 'RandomForest')]

results_list = []

for ds_name, X, y in datasets:
    print(f"\nДатасет: {ds_name}")
    for m_code, m_name in models_to_tune:
        start_time = time.time()
        
        best_pos, score = run_pso_tuning(m_code, X, y)
        duration = time.time() - start_time

        results_list.append({
            "Dataset": ds_name,
            "Model": m_name,
            "F1 Score": round(score, 4),
            "Duration": round(duration, 4)
        })
        
        if m_code == 'KNN':
            final_model = KNeighborsClassifier(n_neighbors=int(best_pos[0]), p=int(best_pos[1])).fit(X, y)
        else:
            final_model = RandomForestClassifier(n_estimators=int(best_pos[0]), max_depth=int(best_pos[1])).fit(X, y)
            
        joblib.dump(final_model, f"best_{m_code}_{ds_name.lower()}_swarm")

df_results = pd.DataFrame(results_list)
df_results

2026-04-13 18:09:37,436 - pyswarms.single.global_best - INFO - Optimize for 10 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9}



Датасет: Small


pyswarms.single.global_best: 100%|██████████|10/10, best_cost=-0.704
2026-04-13 18:09:46,014 - pyswarms.single.global_best - INFO - Optimization finished | best cost: -0.7039467068878834, best pos: [42.34619345  4.29266403]
2026-04-13 18:09:46,019 - pyswarms.single.global_best - INFO - Optimize for 10 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9}
pyswarms.single.global_best: 100%|██████████|10/10, best_cost=-0.693
2026-04-13 18:09:54,865 - pyswarms.single.global_best - INFO - Optimization finished | best cost: -0.6928391869568341, best pos: [35.71775242 10.26730374]
2026-04-13 18:09:54,888 - pyswarms.single.global_best - INFO - Optimize for 10 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9}



Датасет: Large


pyswarms.single.global_best: 100%|██████████|10/10, best_cost=-0.624
2026-04-13 18:10:20,353 - pyswarms.single.global_best - INFO - Optimization finished | best cost: -0.6239003057212777, best pos: [82.75587673 10.86425807]
2026-04-13 18:10:20,358 - pyswarms.single.global_best - INFO - Optimize for 10 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9}
pyswarms.single.global_best: 100%|██████████|10/10, best_cost=-0.626
2026-04-13 18:11:00,590 - pyswarms.single.global_best - INFO - Optimization finished | best cost: -0.6257637247486367, best pos: [181.04614785  10.4322867 ]


,Dataset,Model,F1 Score,Duration
0,Small,KNeighbors,0.7039,8.5872
1,Small,RandomForest,0.6928,8.8497
2,Large,KNeighbors,0.6239,25.4693
3,Large,RandomForest,0.6258,40.2351


| Dataset | Model | F1 Score | Fit duration |
|---------|-------|----------|---------------|
| Small | KNeighborsClassifier | 0.600 | 0.610465 |
| Small | RandomForestClassifier | 0.680 | 4.116801 |
| Big | KNeighborsClassifier | 0.693 | 2.531045 |
| Big | RandomForestClassifier | 0.656 | 10.344154 |